 Install and Import

In [1]:
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])

import pandas as pd
import spacy
import json
import random
from spacy.training import Example
from spacy.util import minibatch, compounding
import warnings
warnings.filterwarnings('ignore')

Load Dataset

In [2]:
df = pd.read_csv('../data/raw/parking_notes.csv')
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100, 4)


,note,zone,floor,landmark
0,Parked near the blue pillar in Zone A,A,NaN,blue pillar
1,I left my car on Level 2 beside the elevator,NaN,2.0,elevator
2,Near the Engineering Building main gate,NaN,NaN,Engineering Building gate
3,Parked in Zone B close to the red wall,B,NaN,red wall
4,Level 3 next to the staircase,NaN,3.0,staircase


Convert Dataset to SpaCy Training Format

In [5]:
def create_training_data(df):
    training_data = []
    for _, row in df.iterrows():
        note = row['note']
        entities = []

        # Find zone in text
        if pd.notna(row['zone']) and str(row['zone']).strip() != '':
            zone_val = str(row['zone']).strip()
            start = note.find(zone_val)
            if start != -1:
                entities.append((start, start + len(zone_val), 'ZONE'))

        # Find floor in text
        if pd.notna(row['floor']) and str(row['floor']).strip() != '':
            floor_val = str(row['floor']).strip()
            start = note.find(floor_val)
            if start != -1:
                entities.append((start, start + len(floor_val), 'FLOOR'))

        # Find landmark in text
        if pd.notna(row['landmark']) and str(row['landmark']).strip() != '':
            landmark_val = str(row['landmark']).strip()
            start = note.find(landmark_val)
            if start != -1:
                entities.append((start, start + len(landmark_val), 'LANDMARK'))

        # --- FIX: Remove overlapping entities ---
        entities = sorted(entities, key=lambda x: x[0])
        filtered = []
        last_end = -1
        for ent in entities:
            if ent[0] >= last_end:
                filtered.append(ent)
                last_end = ent[1]

        if filtered:
            training_data.append((note, {'entities': filtered}))

    return training_data

train_data = create_training_data(df)
print(f"Training examples created: {len(train_data)}")
print("\nSample:")
print(train_data[0])

Training examples created: 99

Sample:
('Parked near the blue pillar in Zone A', {'entities': [(16, 27, 'LANDMARK'), (36, 37, 'ZONE')]})


 Train SpaCy NER Model

In [6]:
# Load blank English model
nlp = spacy.blank("en")

# Add NER pipeline
ner = nlp.add_pipe("ner")
ner.add_label("ZONE")
ner.add_label("FLOOR")
ner.add_label("LANDMARK")

# Training
optimizer = nlp.begin_training()
random.seed(42)

print("Training SpaCy NER model...")
for epoch in range(30):
    random.shuffle(train_data)
    losses = {}
    batches = minibatch(train_data, size=compounding(4.0, 32.0, 1.001))

    for batch in batches:
        examples = []
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)
            examples.append(example)
        nlp.update(examples, sgd=optimizer, losses=losses)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} - Loss: {losses['ner']:.3f}")

print("Training complete")

Training SpaCy NER model...
Epoch 10 - Loss: 2.000
Epoch 20 - Loss: 0.000
Epoch 30 - Loss: 0.000
Training complete


Test the SpaCy Model

In [7]:
test_notes = [
    "I parked near the blue sign on Level 3",
    "Zone B beside the elevator on Level 2",
    "Near the library entrance",
    "Parked close to the red pillar in Zone A",
    "Level 4 near the emergency exit"
]

print("SpaCy NER Predictions:")
print("=" * 50)
for note in test_notes:
    doc = nlp(note)
    result = {ent.label_: ent.text for ent in doc.ents}
    print(f"Input   : {note}")
    print(f"Entities: {result}")
    print()

SpaCy NER Predictions:
Input   : I parked near the blue sign on Level 3
Entities: {'LANDMARK': 'blue sign'}

Input   : Zone B beside the elevator on Level 2
Entities: {'ZONE': 'B', 'LANDMARK': 'elevator'}

Input   : Near the library entrance
Entities: {'LANDMARK': 'library entrance'}

Input   : Parked close to the red pillar in Zone A
Entities: {'LANDMARK': 'red pillar', 'ZONE': 'A'}

Input   : Level 4 near the emergency exit
Entities: {'LANDMARK': 'emergency exit'}



 Evaluate SpaCy with F1 Score

In [8]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for note, annotations in train_data[:30]:
    doc = nlp(note)
    pred_entities = {ent.label_: ent.text for ent in doc.ents}

    for start, end, label in annotations['entities']:
        true_text = note[start:end]
        y_true.append(label)
        if label in pred_entities:
            y_pred.append(label)
        else:
            y_pred.append('O')

print("SpaCy NER Evaluation:")
print(classification_report(y_true, y_pred, zero_division=0))

SpaCy NER Evaluation:
              precision    recall  f1-score   support

    LANDMARK       1.00      1.00      1.00        30
        ZONE       1.00      1.00      1.00        11

    accuracy                           1.00        41
   macro avg       1.00      1.00      1.00        41
weighted avg       1.00      1.00      1.00        41



Save SpaCy Model

In [9]:
nlp.to_disk('../models/spacy_ner_model')
print("SpaCy model saved to models/spacy_ner_model")

SpaCy model saved to models/spacy_ner_model
